In [0]:
path = "/Volumes/workspace/dbsf_2305122/raw_files/sales_sample.csv"

df_infer = spark.read.csv(path, header=True, inferSchema=True)
df_infer.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)



In [0]:
df_raw = spark.read.csv(path, header=True, inferSchema=False)
df_raw.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)



In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DateType, DoubleType
)

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("region", StringType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
])

df = spark.read.csv(path, header=True, schema=schema)

df.printSchema()
print(df.count(), "rows")

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)

12 rows


In [0]:
json_path = "/Volumes/workspace/dbsf_2305122/raw_files/sales_json"

df.write.mode("overwrite").json(json_path)

In [0]:
%fs ls /Volumes/workspace/dbsf_2305122/raw_files/sales_json/

path,name,size,modificationTime
dbfs:/Volumes/workspace/dbsf_2305122/raw_files/sales_json/_SUCCESS,_SUCCESS,0,1786273256000
dbfs:/Volumes/workspace/dbsf_2305122/raw_files/sales_json/_committed_3847576007908400973,_committed_3847576007908400973,114,1786273256000
dbfs:/Volumes/workspace/dbsf_2305122/raw_files/sales_json/_started_3847576007908400973,_started_3847576007908400973,0,1786273255000
dbfs:/Volumes/workspace/dbsf_2305122/raw_files/sales_json/part-00000-tid-3847576007908400973-a2067202-1a44-4bdc-852f-3cf3a2b58bbc-141-1-c000.json,part-00000-tid-3847576007908400973-a2067202-1a44-4bdc-852f-3cf3a2b58bbc-141-1-c000.json,1371,1786273255000


In [0]:
df_json = spark.read.json(json_path)

df_json.printSchema()
print(df_json.count(), "rows")

root
 |-- order_date: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- region: string (nullable = true)
 |-- unit_price: double (nullable = true)

12 rows


### Schema Comparison

The following columns did not retain their original types after the JSON round trip:

- `order_id`: `integer` → `long`
- `quantity`: `integer` → `long`
- `order_date`: `date` → `string`


In [0]:
parquet_path = "/Volumes/workspace/dbsf_2305122/raw_files/sales_parquet"

df.write.mode("overwrite").parquet(parquet_path)

In [0]:
%fs ls /Volumes/workspace/dbsf_2305122/raw_files/sales_parquet/

path,name,size,modificationTime
dbfs:/Volumes/workspace/dbsf_2305122/raw_files/sales_parquet/_SUCCESS,_SUCCESS,0,1786274412000
dbfs:/Volumes/workspace/dbsf_2305122/raw_files/sales_parquet/_committed_4713632425653346975,_committed_4713632425653346975,234,1786274412000
dbfs:/Volumes/workspace/dbsf_2305122/raw_files/sales_parquet/_committed_6891221471277132887,_committed_6891221471277132887,124,1786274163000
dbfs:/Volumes/workspace/dbsf_2305122/raw_files/sales_parquet/_started_4713632425653346975,_started_4713632425653346975,0,1786274412000
dbfs:/Volumes/workspace/dbsf_2305122/raw_files/sales_parquet/_started_6891221471277132887,_started_6891221471277132887,0,1786274162000
dbfs:/Volumes/workspace/dbsf_2305122/raw_files/sales_parquet/part-00000-tid-4713632425653346975-1d0bb825-90a5-4296-b4b7-ca80e8a3708c-154-1.c000.snappy.parquet,part-00000-tid-4713632425653346975-1d0bb825-90a5-4296-b4b7-ca80e8a3708c-154-1.c000.snappy.parquet,2183,1786274412000


In [0]:
df_parquet = spark.read.parquet(parquet_path)

df_parquet.printSchema()
print(df_parquet.count(), "rows")

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)

12 rows


### Parquet Schema Comparison

The schema remained unchanged after the Parquet round trip. All column data types were preserved, including `order_id` as `integer`, `order_date` as `date`, `quantity` as `integer`, and `unit_price` as `double`.

## Format Comparison

| Format | Readable as text | Schema survives round trip | Total size on disk |
|---|---|---|---|
| CSV | Yes | No | 546 B |
| JSON | Yes | No | 1371 B |
| Parquet | No | Yes | 2183 B |

### Question 1
If I had to hand this data to someone who has no Spark or Databricks, I would send CSV because it is human-readable and can be opened with common tools.

### Question 2
If the same data had ten million rows, I would store it as Parquet because it is columnar, compressed, and preserves the schema efficiently.

In [0]:
files = dbutils.fs.ls("/Volumes/workspace/dbsf_2305122/raw_files/")

for f in files:
    print(f.name, "->", f.size, "bytes")

sales_json/ -> 0 bytes
sales_parquet/ -> 0 bytes
sales_sample.csv -> 546 bytes
